In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# Load Bronze table
# ---------------------------------------------------------

bronze = spark.table("financial_balance_sheets_bronze")

print("Bronze rows:", bronze.count())
print("Bronze columns:", len(bronze.columns))

display(bronze.limit(10))

In [0]:
from pyspark.sql import functions as F

# ---------------------------------------------------------
# Bronze -> Silver transformation
# ---------------------------------------------------------

silver = (
    bronze
    .select(
        "freq",
        "unit",
        "unit_label",
        "co_nco",
        "co_nco_label",
        "sector",
        "sector_label",
        "finpos",
        "finpos_label",
        "na_item",
        "na_item_label",
        "geo",
        "geo_label",
        "time",
        "value",
        "status",
        "source",
        "dataset",
        "source_url",
        "ingestion_timestamp"
    )
    # Standardise column names / data types
    .withColumn("reference_year", F.col("time").cast("int"))
    .withColumn("is_observed", F.col("value").isNotNull())
    # Stable technical key for the observation
    .withColumn(
        "observation_key",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("freq"),
                F.col("unit"),
                F.col("co_nco"),
                F.col("sector"),
                F.col("finpos"),
                F.col("na_item"),
                F.col("geo"),
                F.col("time")
            ),
            256
        )
    )
)

print("Silver rows:", silver.count())
print("Silver columns:", len(silver.columns))

silver.printSchema()

display(silver.limit(20))

In [0]:
# ---------------------------------------------------------
# Save Silver Delta table
# ---------------------------------------------------------

table_name = "financial_balance_sheets_silver"

(
    silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(table_name)
)

print(
    f"Silver table created successfully: {table_name}"
)

In [0]:
# ---------------------------------------------------------
# Verify Silver table
# ---------------------------------------------------------

silver_table = spark.table(
    "financial_balance_sheets_silver"
)

print("Rows:", silver_table.count())
print("Columns:", len(silver_table.columns))

display(
    silver_table.limit(20)
)